# Market-Driven Skill Recommendation: Optimized Modeling Notebook

Notebook này giữ nguyên **7 file đầu vào** và logic so sánh theo proposal:

- Popularity
- FP-Growth only
- Node2Vec-style graph embedding only
- Hybrid

Mục tiêu của bản này là:

- chạy ổn định theo đúng thứ tự cell
- không phụ thuộc vào biến global dễ vỡ
- tối ưu hiệu năng ở các bước recommendation và evaluation

## 1. Install packages

Chạy cell này một lần trên Colab.  
Nếu chạy local và đã cài đủ package thì có thể bỏ qua.

In [1]:
import sys

!{sys.executable} -m pip install -q setuptools
!{sys.executable} -m pip install -q "gensim>=4.3.0,<5.0.0" "joblib>=1.4.0" "networkx==3.3" "tqdm>=4.29.0" "mlxtend>=0.23.1"


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports and global config

In [1]:
# Imports
import ast
import math
import os
import random
from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import networkx as nx

from gensim.models import Word2Vec
from mlxtend.frequent_patterns import fpgrowth, association_rules

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
pd.set_option("display.max_colwidth", 200)

## 3. Upload 7 CSV files

## 4. Load data

In [2]:
clean_df = pd.read_csv("cleaned_model_population.csv")
vocab_df = pd.read_csv("skill_vocabulary_final.csv")
train_df = pd.read_csv("train_users.csv")
valid_df = pd.read_csv("valid_users.csv")
test_df = pd.read_csv("test_users.csv")
train_matrix = pd.read_csv("fpgrowth_train_matrix.csv")
edges_df = pd.read_csv("graph_edges_train.csv")

print("clean_df:", clean_df.shape)
print("vocab_df:", vocab_df.shape)
print("train_df:", train_df.shape)
print("valid_df:", valid_df.shape)
print("test_df:", test_df.shape)
print("train_matrix:", train_matrix.shape)
print("edges_df:", edges_df.shape)

display(train_df.head(2))

clean_df: (24229, 13)
vocab_df: (181, 7)
train_df: (19383, 13)
valid_df: (2423, 13)
test_df: (2423, 13)
train_matrix: (19383, 181)
edges_df: (16290, 3)


,ResponseId,MainBranch,Employment,WorkExp,WorkExp_num,DevType,is_junior_0_3y,have_skills,want_skills,target_skills,n_have,n_want,n_target
0,42896,I am a developer by profession,Employed,2.0,2.0,"Developer, full-stack",True,Language:C#|Language:HTML/CSS|Language:JavaScript|Language:SQL|Language:TypeScript|Language:Visual Basic (.Net)|Database:Microsoft SQL Server|Webframe:ASP.NET|Webframe:React|DevEnv:Notepad++|DevEn...,Language:MATLAB|Language:Python|DevEnv:PyCharm,Language:MATLAB|Language:Python|DevEnv:PyCharm,12,3,3
1,36268,I am a developer by profession,Employed,8.0,8.0,"Developer, embedded applications or devices",False,Language:Assembly|Language:Bash/Shell (all shells)|Language:C|Language:Java|Language:JavaScript|Language:Perl|Language:PowerShell|Language:Python|Platform:Make|Platform:Maven (build tool)|Platform...,Language:Bash/Shell (all shells)|Language:C|Language:Python|Platform:Make|Platform:Pacman|Platform:Pip|DevEnv:Visual Studio Code,NaN,17,7,0


## 5. Validate schema and build model-ready lists

In [3]:
REQUIRED_SPLIT_COLS = ["have_skills", "want_skills", "target_skills"]
for df_name, df in [("train_df", train_df), ("valid_df", valid_df), ("test_df", test_df)]:
    missing = [c for c in REQUIRED_SPLIT_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"{df_name} thiếu cột: {missing}")

if "skill" not in vocab_df.columns:
    raise ValueError("skill_vocabulary_final.csv phải có cột 'skill'")

def parse_skill_pipe_list(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if text == "":
        return []
    return [item.strip() for item in text.split("|") if item.strip()]

def attach_skill_lists(df):
    out = df.copy()
    out["have_list"] = out["have_skills"].apply(parse_skill_pipe_list)
    out["want_list"] = out["want_skills"].apply(parse_skill_pipe_list)
    out["target_list"] = out["target_skills"].apply(parse_skill_pipe_list)
    return out

train_df = attach_skill_lists(train_df)
valid_df = attach_skill_lists(valid_df)
test_df = attach_skill_lists(test_df)

skill_catalog = set(vocab_df["skill"].astype(str).tolist())
matrix_skill_catalog = set(train_matrix.columns.astype(str).tolist())

# Dùng intersection để đảm bảo catalog khớp hoàn toàn với feature space train
skill_catalog = skill_catalog & matrix_skill_catalog if len(skill_catalog) > 0 else matrix_skill_catalog

assert "have_list" in train_df.columns
assert "have_list" in valid_df.columns
assert "have_list" in test_df.columns

print("Skill catalog size:", len(skill_catalog))
print("Train users with non-empty target:", int((train_df["target_list"].apply(len) > 0).sum()))
print("Valid users with non-empty target:", int((valid_df["target_list"].apply(len) > 0).sum()))
print("Test users with non-empty target:", int((test_df["target_list"].apply(len) > 0).sum()))
display(train_df[["ResponseId", "have_list", "target_list"]].head(2))

Skill catalog size: 181
Train users with non-empty target: 13295
Valid users with non-empty target: 1629
Test users with non-empty target: 1701


,ResponseId,have_list,target_list
0,42896,"[Language:C#, Language:HTML/CSS, Language:JavaScript, Language:SQL, Language:TypeScript, Language:Visual Basic (.Net), Database:Microsoft SQL Server, Webframe:ASP.NET, Webframe:React, DevEnv:Notep...","[Language:MATLAB, Language:Python, DevEnv:PyCharm]"
1,36268,"[Language:Assembly, Language:Bash/Shell (all shells), Language:C, Language:Java, Language:JavaScript, Language:Perl, Language:PowerShell, Language:Python, Platform:Make, Platform:Maven (build tool...",[]


## 6. Experiment config

In [4]:
CONFIG = {
    "fp_min_support": 0.03,
    "fp_min_confidence": 0.30,
    "fp_max_antecedent_len": 3,
    "n2v_dim": 64,
    "n2v_walk_length": 12,
    "n2v_num_walks": 20,
    "n2v_window": 5,
    "n2v_epochs": 4,
    "n2v_p": 1.0,
    "n2v_q": 0.5,
    "sim_cache_topn": 30,
    "alpha": 0.6,
    "beta": 0.4,
}
CONFIG

{'fp_min_support': 0.03,
 'fp_min_confidence': 0.3,
 'fp_max_antecedent_len': 3,
 'n2v_dim': 64,
 'n2v_walk_length': 12,
 'n2v_num_walks': 20,
 'n2v_window': 5,
 'n2v_epochs': 4,
 'n2v_p': 1.0,
 'n2v_q': 0.5,
 'sim_cache_topn': 30,
 'alpha': 0.6,
 'beta': 0.4}

## 7. Evaluation utilities

In [5]:
def recall_at_k(predictions, target_skills, k=10):
    target_set = set(target_skills)
    if len(target_set) == 0:
        return np.nan
    return len(set(predictions[:k]) & target_set) / len(target_set)

def hitrate_at_k(predictions, target_skills, k=10):
    target_set = set(target_skills)
    if len(target_set) == 0:
        return np.nan
    return 1.0 if len(set(predictions[:k]) & target_set) > 0 else 0.0

def diversity_at_k(predictions, k=10):
    top_k_predictions = predictions[:k]
    if len(top_k_predictions) <= 1:
        return 0.0
    return len(set(top_k_predictions)) / len(top_k_predictions)

def evaluate_predictions(df, pred_col, k=10, catalog=None):
    if catalog is None:
        raise ValueError("catalog must be provided to evaluate_predictions()")

    eval_df = df[df["target_list"].apply(len) > 0].copy()

    if len(eval_df) == 0:
        return {
            "UsersEvaluated": 0,
            "Recall@K": np.nan,
            "HitRate@K": np.nan,
            "CatalogCoverage": np.nan,
            "Diversity@K": np.nan,
        }

    eval_df["recall"] = eval_df.apply(
        lambda row: recall_at_k(row[pred_col], row["target_list"], k=k),
        axis=1,
    )
    eval_df["hit"] = eval_df.apply(
        lambda row: hitrate_at_k(row[pred_col], row["target_list"], k=k),
        axis=1,
    )
    eval_df["diversity"] = eval_df[pred_col].apply(lambda items: diversity_at_k(items, k=k))

    recommended_skills = set()
    for items in eval_df[pred_col]:
        recommended_skills.update(items[:k])

    catalog_coverage = len(recommended_skills) / max(len(catalog), 1)

    return {
        "UsersEvaluated": int(len(eval_df)),
        "Recall@K": float(np.nanmean(eval_df["recall"])),
        "HitRate@K": float(np.nanmean(eval_df["hit"])),
        "CatalogCoverage": float(catalog_coverage),
        "Diversity@K": float(np.nanmean(eval_df["diversity"])),
    }

## 8. Popularity baseline

In [6]:
skill_support_counter = Counter()
for user_skills in train_df["have_list"]:
    skill_support_counter.update(set(user_skills))

global_popularity_ranking = [skill for skill, _ in skill_support_counter.most_common()]
skill_support_fraction = {
    skill: count / max(len(train_df), 1)
    for skill, count in skill_support_counter.items()
}

def recommend_popularity(user_have, k=10):
    seen_skills = set(user_have)
    recommendations = []
    for skill in global_popularity_ranking:
        if skill not in seen_skills:
            recommendations.append(skill)
        if len(recommendations) == k:
            break
    return recommendations

for df in [valid_df, test_df]:
    df["pred_popularity_10"] = df["have_list"].apply(lambda skills: recommend_popularity(skills, k=10))
    df["pred_popularity_5"] = df["pred_popularity_10"].apply(lambda items: items[:5])

## 9. Train FP-Growth exactly once

In [7]:
bool_train_matrix = train_matrix.astype(bool)

# 1) Mine frequent itemsets
fp_itemsets_df = fpgrowth(
    bool_train_matrix,
    min_support=CONFIG["fp_min_support"],
    use_colnames=True,
).copy()

print("Frequent itemsets (raw):", len(fp_itemsets_df))

# 2) Giới hạn kích thước itemset (gián tiếp giới hạn antecedent)
# antecedent_len <= L và consequent_len == 1  => itemset_len <= L + 1
max_itemset_len = CONFIG["fp_max_antecedent_len"] + 1

fp_itemsets_df["itemset_len"] = fp_itemsets_df["itemsets"].apply(len)
fp_itemsets_df = fp_itemsets_df[
    fp_itemsets_df["itemset_len"] <= max_itemset_len
].copy()

print("Frequent itemsets (after length filter):", len(fp_itemsets_df))

# 3) Generate rules
if len(fp_itemsets_df) == 0:
    fp_rules_df = pd.DataFrame()
else:
    fp_rules_df = association_rules(
        fp_itemsets_df,
        metric="confidence",
        min_threshold=CONFIG["fp_min_confidence"],
    ).copy()

# 4) Lọc rules theo antecedent_len và consequent_len
if len(fp_rules_df) > 0:
    fp_rules_df["antecedent_len"] = fp_rules_df["antecedents"].apply(len)
    fp_rules_df["consequent_len"] = fp_rules_df["consequents"].apply(len)

    fp_rules_df = fp_rules_df[
        (fp_rules_df["antecedent_len"] <= CONFIG["fp_max_antecedent_len"]) &
        (fp_rules_df["consequent_len"] == 1)
    ].copy()

    fp_rules_df["consequent_skill"] = fp_rules_df["consequents"].apply(lambda s: list(s)[0])
    fp_rules_df["rule_score"] = fp_rules_df["confidence"] * fp_rules_df["lift"]

    fp_rules_df = fp_rules_df.sort_values(
        ["rule_score", "confidence", "lift", "support"],
        ascending=False,
    ).reset_index(drop=True)

print("Association rules:", len(fp_rules_df))
display(fp_rules_df.head(10))

Frequent itemsets (raw): 429643
Frequent itemsets (after length filter): 81250
Association rules: 255838


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len,consequent_len,consequent_skill,rule_score
0,"frozenset({Language:JavaScript, Platform:Supabase})",frozenset({Database:Supabase}),0.036527,0.048960,0.030026,0.822034,16.789761,1.0,0.028238,5.343937,0.976094,0.541395,0.812872,0.717656,2,1,Database:Supabase,13.801753
1,frozenset({Platform:Supabase}),frozenset({Database:Supabase}),0.041841,0.048960,0.033792,0.807645,16.495870,1.0,0.031744,4.944186,0.980399,0.592760,0.797742,0.748923,1,1,Database:Supabase,13.322805
2,"frozenset({Database:PostgreSQL, Language:JavaScript, Webframe:Ruby on Rails})",frozenset({Language:Ruby}),0.034566,0.068668,0.032038,0.926866,13.497699,1.0,0.029665,12.734534,0.959065,0.450000,0.921473,0.696716,3,1,Language:Ruby,12.510554
3,"frozenset({Language:JavaScript, Language:HTML/CSS, Webframe:Ruby on Rails})",frozenset({Language:Ruby}),0.034928,0.068668,0.032193,0.921713,13.422668,1.0,0.029795,11.896443,0.958994,0.450867,0.915941,0.695267,3,1,Language:Ruby,12.371854
4,"frozenset({Language:SQL, Webframe:Ruby on Rails})",frozenset({Language:Ruby}),0.034308,0.068668,0.031522,0.918797,13.380197,1.0,0.029167,11.469176,0.958135,0.441155,0.912810,0.688925,2,1,Language:Ruby,12.293685
5,"frozenset({Language:JavaScript, Webframe:Ruby on Rails})",frozenset({Language:Ruby}),0.041273,0.068668,0.037868,0.917500,13.361309,1.0,0.035034,11.288868,0.964985,0.525412,0.911417,0.734483,2,1,Language:Ruby,12.259001
6,"frozenset({Database:PostgreSQL, Webframe:Ruby on Rails})",frozenset({Language:Ruby}),0.040190,0.068668,0.036836,0.916560,13.347616,1.0,0.034077,11.161651,0.963816,0.511461,0.910408,0.726499,2,1,Language:Ruby,12.233887
7,"frozenset({Language:HTML/CSS, Webframe:Ruby on Rails})",frozenset({Language:Ruby}),0.037146,0.068668,0.033896,0.912500,13.288495,1.0,0.031345,10.643789,0.960423,0.471306,0.906048,0.703057,2,1,Language:Ruby,12.125752
8,"frozenset({Language:JavaScript, Database:Supabase})",frozenset({Platform:Supabase}),0.042408,0.041841,0.030026,0.708029,16.921985,1.0,0.028252,3.281695,0.982575,0.553758,0.695279,0.712831,2,1,Platform:Supabase,11.981260
9,"frozenset({Platform:Docker, Webframe:Ruby on Rails})",frozenset({Language:Ruby}),0.037198,0.068668,0.033689,0.905687,13.189273,1.0,0.031135,9.874854,0.959886,0.466762,0.898733,0.698148,2,1,Language:Ruby,11.945347


## 10. Build optimized FP rule index

In [8]:
def build_rule_index_by_len(rules_df):
    rule_index = defaultdict(dict)
    max_antecedent_len = 0

    if rules_df is None or len(rules_df) == 0:
        return rule_index, max_antecedent_len

    for _, row in rules_df.iterrows():
        antecedent = frozenset(row["antecedents"])
        antecedent_len = len(antecedent)
        max_antecedent_len = max(max_antecedent_len, antecedent_len)

        if antecedent not in rule_index[antecedent_len]:
            rule_index[antecedent_len][antecedent] = []

        rule_index[antecedent_len][antecedent].append({
            "skill": row["consequent_skill"],
            "rule_score": float(row["rule_score"]),
            "support": float(row["support"]),
            "confidence": float(row["confidence"]),
            "lift": float(row["lift"]),
        })

    return rule_index, max_antecedent_len

rule_index_by_len, rule_index_max_len = build_rule_index_by_len(fp_rules_df)
print("Rule buckets:", {k: len(v) for k, v in rule_index_by_len.items()})
print("Max antecedent length in rules:", rule_index_max_len)

Rule buckets: {2: 2593, 1: 130, 3: 16599}
Max antecedent length in rules: 3


## 11. FP-Growth recommendation

In [9]:
def recommend_fpgrowth(user_have, rule_index_by_len, k=10, max_ant_len=3):
    seen_skills = set(user_have)
    user_skill_list = sorted(seen_skills)
    candidate_scores = {}

    if len(user_skill_list) == 0:
        return []

    effective_max_len = min(max_ant_len, len(user_skill_list))

    for antecedent_len in range(1, effective_max_len + 1):
        bucket = rule_index_by_len.get(antecedent_len, {})
        if not bucket:
            continue

        for comb in combinations(user_skill_list, antecedent_len):
            antecedent = frozenset(comb)
            if antecedent not in bucket:
                continue

            for rule_info in bucket[antecedent]:
                skill = rule_info["skill"]
                if skill in seen_skills:
                    continue

                score = rule_info["rule_score"]
                old_score = candidate_scores.get(skill, -1.0)
                if score > old_score:
                    candidate_scores[skill] = score

    ranked = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)
    return [skill for skill, _ in ranked[:k]]

effective_fp_ant_len = min(CONFIG["fp_max_antecedent_len"], max(rule_index_max_len, 1))

for df in [valid_df, test_df]:
    df["pred_fp_10"] = df["have_list"].apply(
        lambda skills: recommend_fpgrowth(skills, rule_index_by_len, k=10, max_ant_len=effective_fp_ant_len)
    )
    df["pred_fp_5"] = df["pred_fp_10"].apply(lambda items: items[:5])

## 12. Build weighted skill graph

In [10]:
edge_a_col = "skill_a" if "skill_a" in edges_df.columns else ("source" if "source" in edges_df.columns else edges_df.columns[0])
edge_b_col = "skill_b" if "skill_b" in edges_df.columns else ("target" if "target" in edges_df.columns else edges_df.columns[1])
edge_w_col = "weight" if "weight" in edges_df.columns else edges_df.columns[2]

skill_graph = nx.Graph()

for _, row in edges_df.iterrows():
    skill_a = str(row[edge_a_col])
    skill_b = str(row[edge_b_col])
    weight = float(row[edge_w_col])

    if skill_a == skill_b:
        continue

    if skill_graph.has_edge(skill_a, skill_b):
        skill_graph[skill_a][skill_b]["weight"] += weight
    else:
        skill_graph.add_edge(skill_a, skill_b, weight=weight)

print("Graph nodes:", skill_graph.number_of_nodes())
print("Graph edges:", skill_graph.number_of_edges())

Graph nodes: 181
Graph edges: 16290


## 13. Deterministic biased random walks

In [11]:
def node2vec_transition_weight(graph, prev_node, current_node, next_node, p=1.0, q=1.0):
    base_weight = graph[current_node][next_node].get("weight", 1.0)

    if prev_node is None:
        return base_weight

    if next_node == prev_node:
        alpha = 1.0 / p
    elif graph.has_edge(next_node, prev_node):
        alpha = 1.0
    else:
        alpha = 1.0 / q

    return base_weight * alpha

def weighted_choice(items, weights, rng):
    total = sum(weights)
    if total <= 0:
        return rng.choice(items)

    threshold = rng.random() * total
    cumulative = 0.0
    for item, weight in zip(items, weights):
        cumulative += weight
        if cumulative >= threshold:
            return item
    return items[-1]

def generate_biased_walk(graph, start_node, walk_length=12, p=1.0, q=1.0, seed=42):
    rng = random.Random(seed)
    walk = [start_node]

    while len(walk) < walk_length:
        current_node = walk[-1]
        neighbors = sorted(list(graph.neighbors(current_node)))
        if len(neighbors) == 0:
            break

        if len(walk) == 1:
            weights = [graph[current_node][nbr].get("weight", 1.0) for nbr in neighbors]
            next_node = weighted_choice(neighbors, weights, rng)
        else:
            prev_node = walk[-2]
            weights = [
                node2vec_transition_weight(graph, prev_node, current_node, nbr, p=p, q=q)
                for nbr in neighbors
            ]
            next_node = weighted_choice(neighbors, weights, rng)

        walk.append(next_node)

    return walk

node_order = {node: idx for idx, node in enumerate(sorted(skill_graph.nodes()))}

def generate_walk_corpus(graph, num_walks=20, walk_length=12, p=1.0, q=1.0, seed=42):
    walk_corpus = []
    ordered_nodes = sorted(graph.nodes())
    shuffle_rng = random.Random(seed)

    for walk_round in range(num_walks):
        shuffled_nodes = ordered_nodes.copy()
        shuffle_rng.shuffle(shuffled_nodes)

        for start_node in shuffled_nodes:
            walk_seed = seed + walk_round * 100000 + node_order[start_node]
            walk = generate_biased_walk(
                graph,
                start_node,
                walk_length=walk_length,
                p=p,
                q=q,
                seed=walk_seed,
            )
            if len(walk) >= 2:
                walk_corpus.append(walk)

    return walk_corpus

walks = generate_walk_corpus(
    skill_graph,
    num_walks=CONFIG["n2v_num_walks"],
    walk_length=CONFIG["n2v_walk_length"],
    p=CONFIG["n2v_p"],
    q=CONFIG["n2v_q"],
    seed=RANDOM_SEED,
)

print("Number of walks:", len(walks))
print("Example walk:", walks[0][:10] if len(walks) > 0 else [])

Number of walks: 3620
Example walk: ['Platform:pnpm', 'Platform:Yarn', 'AIModel:openAI Image generating models', 'Database:Dynamodb', 'Language:Erlang', 'DevEnv:Neovim', 'Platform:Vite', 'Platform:Pip', 'Webframe:Django', 'Language:SQL']


## 14. Train Word2Vec embeddings

In [12]:
skill_embedding_model = Word2Vec(
    sentences=walks,
    vector_size=CONFIG["n2v_dim"],
    window=CONFIG["n2v_window"],
    min_count=1,
    sg=1,
    workers=1,
    epochs=CONFIG["n2v_epochs"],
    seed=RANDOM_SEED,
)

print("Embedding vocabulary size:", len(skill_embedding_model.wv.index_to_key))

Embedding vocabulary size: 181


## 15. Similarity cache

In [13]:
def build_similarity_cache(model, topn=30):
    similarity_cache = {}
    vocab = model.wv.index_to_key
    topn = min(topn, max(len(vocab) - 1, 1))

    for skill in vocab:
        try:
            similarity_cache[skill] = model.wv.most_similar(skill, topn=topn)
        except KeyError:
            similarity_cache[skill] = []

    return similarity_cache

similarity_cache = build_similarity_cache(
    skill_embedding_model,
    topn=CONFIG["sim_cache_topn"],
)

print("Similarity cache size:", len(similarity_cache))

Similarity cache size: 181


## 16. Node2Vec recommendation

In [14]:
def recommend_node2vec_top10(user_have, similarity_cache):
    seen_skills = set(user_have)
    aggregate_scores = defaultdict(float)
    aggregate_counts = defaultdict(int)

    for skill in user_have:
        if skill not in similarity_cache:
            continue

        for other_skill, sim in similarity_cache[skill]:
            if other_skill in seen_skills:
                continue
            aggregate_scores[other_skill] += float(sim)
            aggregate_counts[other_skill] += 1

    for skill in list(aggregate_scores.keys()):
        aggregate_scores[skill] /= aggregate_counts[skill]

    ranked = sorted(aggregate_scores.items(), key=lambda x: x[1], reverse=True)
    return [skill for skill, _ in ranked[:10]]

for df in [valid_df, test_df]:
    df["pred_n2v_10"] = df["have_list"].apply(lambda skills: recommend_node2vec_top10(skills, similarity_cache))
    df["pred_n2v_5"] = df["pred_n2v_10"].apply(lambda items: items[:5])

## 17. Hybrid recommendation

In [15]:
def collect_rule_component(user_have, rule_index_by_len, max_ant_len=3):
    seen_skills = set(user_have)
    user_skill_list = sorted(seen_skills)
    rule_component = {}
    rule_support = {}

    if len(user_skill_list) == 0:
        return rule_component, rule_support

    effective_max_len = min(max_ant_len, len(user_skill_list))

    for antecedent_len in range(1, effective_max_len + 1):
        bucket = rule_index_by_len.get(antecedent_len, {})
        if not bucket:
            continue

        for comb in combinations(user_skill_list, antecedent_len):
            antecedent = frozenset(comb)
            if antecedent not in bucket:
                continue

            for rule_info in bucket[antecedent]:
                skill = rule_info["skill"]
                if skill in seen_skills:
                    continue

                score = rule_info["rule_score"]
                support = rule_info["support"]

                if skill not in rule_component or score > rule_component[skill]:
                    rule_component[skill] = score
                    rule_support[skill] = support

    return rule_component, rule_support

def collect_embedding_component(user_have, similarity_cache):
    seen_skills = set(user_have)
    embed_scores = defaultdict(float)
    embed_counts = defaultdict(int)

    for skill in user_have:
        if skill not in similarity_cache:
            continue

        for other_skill, sim in similarity_cache[skill]:
            if other_skill in seen_skills:
                continue
            embed_scores[other_skill] += float(sim)
            embed_counts[other_skill] += 1

    for skill in list(embed_scores.keys()):
        embed_scores[skill] /= embed_counts[skill]

    return dict(embed_scores)

def recommend_hybrid(user_have, rule_index_by_len, similarity_cache, support_fraction_map, alpha=0.6, beta=0.4, k=10, max_ant_len=3):
    seen_skills = set(user_have)

    rule_component, rule_support = collect_rule_component(
        user_have,
        rule_index_by_len,
        max_ant_len=max_ant_len,
    )
    embed_component = collect_embedding_component(user_have, similarity_cache)

    candidate_pool = set(rule_component.keys()) | set(embed_component.keys())

    final_scores = {}
    for skill in candidate_pool:
        if skill in seen_skills:
            continue

        rule_score = rule_component.get(skill, 0.0)
        embed_score = embed_component.get(skill, 0.0)

        combined_score = alpha * rule_score + beta * embed_score

        # popularity normalization
        support_value = support_fraction_map.get(skill, rule_support.get(skill, 1e-9))
        normalized_score = combined_score / max(math.log1p(1.0 + support_value), 1e-9)

        final_scores[skill] = normalized_score

    ranked = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
    return [skill for skill, _ in ranked[:k]]

for df in [valid_df, test_df]:
    df["pred_hybrid_10"] = df["have_list"].apply(
        lambda skills: recommend_hybrid(
            skills,
            rule_index_by_len,
            similarity_cache,
            skill_support_fraction,
            alpha=CONFIG["alpha"],
            beta=CONFIG["beta"],
            k=10,
            max_ant_len=effective_fp_ant_len,
        )
    )
    df["pred_hybrid_5"] = df["pred_hybrid_10"].apply(lambda items: items[:5])

## 18. Compare models

In [16]:
def compare_models(df, k=10):
    prediction_mapping = {
        "Popularity": f"pred_popularity_{k}",
        "FP-Growth only": f"pred_fp_{k}",
        "Node2Vec only": f"pred_n2v_{k}",
        "Hybrid": f"pred_hybrid_{k}",
    }

    result_rows = []
    for model_name, pred_col in prediction_mapping.items():
        metrics = evaluate_predictions(df, pred_col, k=k, catalog=skill_catalog)
        result_rows.append({"Model": model_name, **metrics})

    result_df = pd.DataFrame(result_rows)
    return result_df.sort_values(
        ["Recall@K", "HitRate@K", "CatalogCoverage", "Diversity@K"],
        ascending=False,
    ).reset_index(drop=True)

valid_compare_5 = compare_models(valid_df, k=5)
valid_compare_10 = compare_models(valid_df, k=10)
test_compare_5 = compare_models(test_df, k=5)
test_compare_10 = compare_models(test_df, k=10)

print("Validation @5")
display(valid_compare_5)

print("Validation @10")
display(valid_compare_10)

print("Test @5")
display(test_compare_5)

print("Test @10")
display(test_compare_10)

Validation @5


,Model,UsersEvaluated,Recall@K,HitRate@K,CatalogCoverage,Diversity@K
0,Popularity,1629,0.100074,0.340086,0.198895,1.000000
1,FP-Growth only,1629,0.083522,0.282382,0.513812,0.998772
2,Hybrid,1629,0.080019,0.275015,0.580110,1.000000
3,Node2Vec only,1629,0.027299,0.127072,0.883978,1.000000


Validation @10


,Model,UsersEvaluated,Recall@K,HitRate@K,CatalogCoverage,Diversity@K
0,Popularity,1629,0.160985,0.451197,0.309392,1.000000
1,FP-Growth only,1629,0.154514,0.436464,0.530387,0.998772
2,Hybrid,1629,0.145889,0.420503,0.657459,1.000000
3,Node2Vec only,1629,0.060509,0.231430,0.917127,1.000000


Test @5


,Model,UsersEvaluated,Recall@K,HitRate@K,CatalogCoverage,Diversity@K
0,Popularity,1701,0.114970,0.379777,0.215470,1.0
1,FP-Growth only,1701,0.083684,0.299824,0.519337,1.0
2,Hybrid,1701,0.077531,0.285714,0.546961,1.0
3,Node2Vec only,1701,0.026141,0.142857,0.889503,1.0


Test @10


,Model,UsersEvaluated,Recall@K,HitRate@K,CatalogCoverage,Diversity@K
0,Popularity,1701,0.180313,0.504409,0.292818,1.0
1,FP-Growth only,1701,0.153602,0.467372,0.530387,1.0
2,Hybrid,1701,0.139663,0.440917,0.591160,1.0
3,Node2Vec only,1701,0.056558,0.265138,0.922652,1.0


## 19. Junior-only comparison (optional)

In [17]:
junior_flag_col = None
for col in ["is_junior_0_3y", "is_junior"]:
    if col in valid_df.columns:
        junior_flag_col = col
        break

if junior_flag_col is not None:
    valid_junior_df = valid_df[valid_df[junior_flag_col] == True].copy()
    test_junior_df = test_df[test_df[junior_flag_col] == True].copy()

    print("Validation junior @10")
    display(compare_models(valid_junior_df, k=10))

    print("Test junior @10")
    display(compare_models(test_junior_df, k=10))
else:
    print("No junior flag column found.")

Validation junior @10


,Model,UsersEvaluated,Recall@K,HitRate@K,CatalogCoverage,Diversity@K
0,Popularity,209,0.182727,0.540670,0.209945,1.0
1,FP-Growth only,209,0.161057,0.478469,0.513812,1.0
2,Hybrid,209,0.139842,0.444976,0.513812,1.0
3,Node2Vec only,209,0.060607,0.296651,0.828729,1.0


Test junior @10


,Model,UsersEvaluated,Recall@K,HitRate@K,CatalogCoverage,Diversity@K
0,Popularity,258,0.188269,0.589147,0.237569,1.0
1,FP-Growth only,258,0.143747,0.511628,0.502762,1.0
2,Hybrid,258,0.132866,0.480620,0.558011,1.0
3,Node2Vec only,258,0.064429,0.344961,0.845304,1.0


## 20. Export outputs for report writing

In [18]:
sample_cases = test_df[test_df["target_list"].apply(len) > 0].copy().head(20)
sample_cases = sample_cases[[
    "ResponseId",
    "have_list",
    "target_list",
    "pred_popularity_10",
    "pred_fp_10",
    "pred_n2v_10",
    "pred_hybrid_10",
]]

valid_compare_5.to_csv("valid_model_comparison_at5.csv", index=False)
valid_compare_10.to_csv("valid_model_comparison_at10.csv", index=False)
test_compare_5.to_csv("test_model_comparison_at5.csv", index=False)
test_compare_10.to_csv("test_model_comparison_at10.csv", index=False)
sample_cases.to_csv("sample_test_recommendations.csv", index=False)

if len(fp_rules_df) > 0:
    fp_rules_df.head(500).to_csv("best_fpgrowth_rules.csv", index=False)
else:
    pd.DataFrame().to_csv("best_fpgrowth_rules.csv", index=False)

print("Saved output files:")
print("- valid_model_comparison_at5.csv")
print("- valid_model_comparison_at10.csv")
print("- test_model_comparison_at5.csv")
print("- test_model_comparison_at10.csv")
print("- sample_test_recommendations.csv")
print("- best_fpgrowth_rules.csv")

display(sample_cases.head(10))

Saved output files:
- valid_model_comparison_at5.csv
- valid_model_comparison_at10.csv
- test_model_comparison_at5.csv
- test_model_comparison_at10.csv
- sample_test_recommendations.csv
- best_fpgrowth_rules.csv


,ResponseId,have_list,target_list,pred_popularity_10,pred_fp_10,pred_n2v_10,pred_hybrid_10
1,10164,"[Language:Bash/Shell (all shells), Language:C, Language:C#, Language:C++, Language:Go, Language:HTML/CSS, Language:JavaScript, Language:PowerShell, Language:Python, Language:Rust, Language:SQL, Da...",[AIModel:openAI Image generating models],"[Language:TypeScript, Platform:npm, Webframe:Node.js, Platform:Amazon Web Services (AWS), Webframe:React, Database:MySQL, Database:SQLite, Language:Java, Database:Redis, Platform:Kubernetes]","[Platform:NuGet, Webframe:ASP.NET Core, Webframe:ASP.NET, Platform:MSBuild, Platform:Microsoft Azure, Platform:Kubernetes, Platform:npm, DevEnv:IntelliJ IDEA, Database:Redis, AIModel:openAI Reason...","[Database:Redis, Webframe:NestJS, AIModel:Perplexity Sonar models, Language:PHP, Webframe:Flask, AIModel:Microsoft Phi-4 models, Webframe:FastAPI, Language:Kotlin, Webframe:Django, Platform:Firebase]","[Platform:NuGet, Webframe:ASP.NET Core, Webframe:ASP.NET, Platform:MSBuild, Platform:Microsoft Azure, Platform:Kubernetes, DevEnv:IntelliJ IDEA, AIModel:openAI Reasoning models, Database:Redis, AI..."
2,44230,"[Language:Bash/Shell (all shells), Language:C, Language:C++, Language:HTML/CSS, Language:PowerShell, Language:Python, Language:SQL, Database:PostgreSQL, DevEnv:Notepad++, DevEnv:Vim, DevEnv:Visual...","[Language:Go, Language:Rust]","[Language:JavaScript, Platform:Docker, Language:TypeScript, Platform:npm, AIModel:openAI GPT (chatbot models), Webframe:Node.js, Platform:Amazon Web Services (AWS), Webframe:React, Database:MySQL,...","[DevEnv:Visual Studio, Database:Microsoft SQL Server, Platform:Make, Platform:Pip, Platform:NuGet, Language:C#, Webframe:ASP.NET Core, Database:SQLite, Platform:Docker, Language:JavaScript]","[Webframe:Flask, Platform:MSBuild, AIModel:openAI Reasoning models, Database:Dynamodb, DevEnv:Jupyter Notebook/JupyterLab, Language:VBA, Webframe:Node.js, Database:Microsoft Access, Language:Elixi...","[DevEnv:Visual Studio, Platform:Make, Webframe:ASP.NET Core, Platform:Pip, Language:C#, Platform:MSBuild, Database:Microsoft SQL Server, Platform:Docker, Platform:npm, Platform:NuGet]"
4,34410,"[Language:Bash/Shell (all shells), Language:HTML/CSS, Language:Java, Language:PowerShell, Language:SQL, Language:TypeScript, Database:Elasticsearch, Database:H2, Database:MongoDB, Database:Postgre...","[Platform:Terraform, DevEnv:IntelliJ IDEA]","[Language:JavaScript, Language:Python, Platform:npm, AIModel:openAI GPT (chatbot models), Webframe:Node.js, Platform:Amazon Web Services (AWS), Webframe:React, Database:MySQL, Database:SQLite, Pla...","[DevEnv:IntelliJ IDEA, Platform:Gradle, Database:MySQL, Webframe:Node.js, DevEnv:Visual Studio, Platform:npm, Database:Microsoft SQL Server, Platform:Kubernetes, Webframe:Express, Platform:Pip]","[Webframe:Flask, AIModel:Anthropic: Claude Sonnet, Platform:Podman, AIModel:Gemini (Pro Reasoning models), AIModel:Alibaba Cloud Qwen models, Language:VBA, Database:Microsoft Access, Database:Clic...","[DevEnv:IntelliJ IDEA, Platform:Gradle, Webframe:Express, DevEnv:Visual Studio, Database:MySQL, Platform:Firebase, Platform:Kubernetes, Webframe:Node.js, Platform:Make, Platform:Pip]"
5,15528,"[Language:C#, Language:HTML/CSS, Language:JavaScript, Database:Microsoft SQL Server, Platform:NuGet, Webframe:ASP.NET Core, Webframe:jQuery, DevEnv:Vim, DevEnv:Visual Studio, DevEnv:Visual Studio ...","[Language:Rust, Platform:Ansible, Platform:Docker, Platform:Kubernetes, Platform:Podman, Platform:Prometheus]","[Language:SQL, Platform:Docker, Language:Python, Database:PostgreSQL, Language:TypeScript, Language:Bash/Shell (all shells), Platform:npm, Webframe:Node.js, Platform:Amazon Web Services (AWS), Web...","[Webframe:ASP.NET, Platform:MSBuild, Platform:Microsoft Azure, Webframe:Blazor, DevEnv:Notepad++, Language:PowerShell, Platform:npm, Language:SQL, Platform:Docker, Platform:Pip]","[Webframe:Flask, Webframe:NestJS, Language:Erlang, DevEnv:Claude Code, Platform:IBM Cloud, 